<a href="https://colab.research.google.com/github/OleksiiLatypov/llm-zoomcamp/blob/main/ml-zoomcamp/Untitled94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://archive.ics.uci.edu/static/public/222/bank+marketing.zip

--2024-10-09 18:46:45--  https://archive.ics.uci.edu/static/public/222/bank+marketing.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘bank+marketing.zip’

bank+marketing.zip      [  <=>               ] 999.85K  2.62MB/s    in 0.4s    

2024-10-09 18:46:46 (2.62 MB/s) - ‘bank+marketing.zip’ saved [1023843]



In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import zipfile

In [3]:
! unzip /content/bank+marketing.zip

Archive:  /content/bank+marketing.zip
 extracting: bank.zip                
 extracting: bank-additional.zip     


In [4]:
! unzip /content/bank.zip

Archive:  /content/bank.zip
  inflating: bank-full.csv           
  inflating: bank-names.txt          
  inflating: bank.csv                


In [130]:
df = pd.read_csv('/content/bank.csv', sep=';')

In [131]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


In [132]:
columns_to_keep = [
    'age', 'job', 'marital', 'education',
    'balance', 'housing', 'contact',
    'day', 'month', 'duration',
    'campaign', 'pdays', 'previous',
    'poutcome', 'y'
]

df = df[columns_to_keep]

age,

job,

marital,

education,

balance,

housing,

contact,

day,

month,

duration,

campaign,

pdays,

previous,

poutcome,

y

In [133]:
df.head()

,age,job,marital,education,balance,housing,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,1787,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,4789,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,1350,yes,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,1476,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,0,yes,unknown,5,may,226,1,-1,0,unknown,no


In [134]:
df.isna().sum()

,0
age,0
job,0
marital,0
education,0
balance,0
housing,0
contact,0
day,0
month,0
duration,0


**Question 1**

What is the most frequent observation (mode) for the column education?

In [135]:
df['education'].mode()[0]

'secondary'

**Question 2**

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

In [136]:
numerical_features = df.select_dtypes(include=['int64', 'float64'])

# Create the correlation matrix
correlation_matrix = numerical_features.corr()
correlation_matrix

,age,balance,day,duration,campaign,pdays,previous
age,1.000000,0.083820,-0.017853,-0.002367,-0.005148,-0.008894,-0.003511
balance,0.083820,1.000000,-0.008677,-0.015950,-0.009976,0.009437,0.026196
day,-0.017853,-0.008677,1.000000,-0.024629,0.160706,-0.094352,-0.059114
duration,-0.002367,-0.015950,-0.024629,1.000000,-0.068382,0.010380,0.018080
campaign,-0.005148,-0.009976,0.160706,-0.068382,1.000000,-0.093137,-0.067833
pdays,-0.008894,0.009437,-0.094352,0.010380,-0.093137,1.000000,0.577562
previous,-0.003511,0.026196,-0.059114,0.018080,-0.067833,0.577562,1.000000


In [137]:
# Find the two features with the highest correlation
max_corr = correlation_matrix.abs().unstack().sort_values(ascending=False)
# Remove self-correlations (where features are the same)
max_corr = max_corr[max_corr < 1]
max_corr[:10]
# # Get the two features with the highest correlation
# highest_corr_features = max_corr.idxmax()
# highest_corr_value = max_corr.max()

# print(f"\nThe two features with the highest correlation are: {highest_corr_features} with a correlation of {highest_corr_value:.4f}")

,,0
previous,pdays,0.577562
pdays,previous,0.577562
day,campaign,0.160706
campaign,day,0.160706
pdays,day,0.094352
day,pdays,0.094352
campaign,pdays,0.093137
pdays,campaign,0.093137
age,balance,0.083820
balance,age,0.083820


Target encoding
Now we want to encode the y variable.
Let's replace the values yes/no with 1/0.

In [138]:
df['y'] = df['y'].replace({'yes': 1, 'no': 0})

<ipython-input-138-9dc15415cc11>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['y'] = df['y'].replace({'yes': 1, 'no': 0})


In [139]:
df.head()

,age,job,marital,education,balance,housing,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,1787,no,cellular,19,oct,79,1,-1,0,unknown,0
1,33,services,married,secondary,4789,yes,cellular,11,may,220,1,339,4,failure,0
2,35,management,single,tertiary,1350,yes,cellular,16,apr,185,1,330,1,failure,0
3,30,management,married,tertiary,1476,yes,unknown,3,jun,199,4,-1,0,unknown,0
4,59,blue-collar,married,secondary,0,yes,unknown,5,may,226,1,-1,0,unknown,0


In [16]:
# y = df['y']
# X = df.drop('y', axis=1)

In [140]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [141]:
df_train.shape, df_test.shape, df_val.shape

((2712, 15), (905, 15), (904, 15))

In [142]:
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

In [143]:
y_train = df_train.y
y_test = df_test.y
y_val = df_val.y

del df_train['y']
del df_val['y']
del df_test['y']

In [144]:
len(y_train)

2712

In [145]:
df_train.head()

,age,job,marital,education,balance,housing,contact,day,month,duration,campaign,pdays,previous,poutcome
0,41,management,married,tertiary,72,yes,unknown,7,may,764,3,-1,0,unknown
1,38,management,married,tertiary,126,yes,unknown,21,may,164,2,-1,0,unknown
2,41,blue-collar,married,secondary,1459,no,unknown,5,jun,82,1,-1,0,unknown
3,57,services,married,secondary,162,yes,unknown,5,may,174,1,-1,0,unknown
4,30,admin.,married,secondary,124,no,telephone,16,jun,161,2,-1,0,unknown


In [146]:
from sklearn.metrics import mutual_info_score

In [147]:
def mutual_info(series):
  return mutual_info_score(series, df_full_train['y'])

In [148]:
categorical_columns = df_full_train.select_dtypes(include=['object', 'category']).columns.tolist()
mi_score = df_full_train[categorical_columns].apply(mutual_info)
mi_score.sort_values(ascending=False)

,0
poutcome,0.028118
month,0.023320
contact,0.010747
job,0.007722
housing,0.006451
marital,0.001893
education,0.001679


**Question 4**

Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.

To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

In [149]:
df_train.__len__()
y_train.__len__()

2712

In [150]:
from sklearn.feature_extraction import DictVectorizer


dv = DictVectorizer(sparse=False)
train_dict = df_train.to_dict(orient='records')
#train_dict
X_train = dv.fit_transform(train_dict)

val_dict = df_val.to_dict(orient='records')
X_val = dv.transform(val_dict)

In [151]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [152]:
y_pred = model.predict(X_val)
y_pred[:10]

array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0])

In [153]:
model.intercept_[0]

-0.9177112557702393

In [154]:
model.coef_[0].round(3)

array([ 6.000e-03, -0.000e+00, -5.200e-02,  1.090e-01,  2.210e-01,
       -1.248e+00,  4.000e-03,  4.000e-03, -3.450e-01,  7.000e-03,
        1.400e-02, -5.930e-01, -2.250e-01, -6.930e-01, -8.000e-03,
       -5.900e-01, -7.040e-01,  8.600e-02,  4.800e-02,  4.530e-01,
        2.500e-02, -2.170e-01,  3.660e-01, -3.540e-01, -2.750e-01,
        2.510e-01, -1.500e-02, -5.890e-01, -3.140e-01,  8.300e-02,
       -7.430e-01,  5.210e-01, -3.500e-01, -8.360e-01, -8.870e-01,
        2.280e-01,  9.650e-01, -6.570e-01, -7.960e-01,  9.900e-01,
        5.650e-01, -1.000e-03, -1.116e+00, -2.700e-02,  1.551e+00,
       -1.326e+00, -1.600e-02])

In [155]:
y_pred = model.predict_proba(X_val)[:, 1]
decision_y = (y_pred >= 0.5)
decision_y[:10]

array([False, False,  True, False, False, False, False, False, False,
       False])

In [156]:
(y_val == decision_y).head()

,y
0,True
1,True
2,True
3,True
4,True


In [157]:
accuracy_model = (y_val == decision_y).mean()
rounded_accuracy = round(accuracy_model, 2)
rounded_accuracy

0.89

In [158]:
accuracy_model

0.8871681415929203

In [100]:
dict(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))

{'age': 0.004,
 'balance': -0.0,
 'campaign': -0.05,
 'contact=cellular': 0.078,
 'contact=telephone': 0.208,
 'contact=unknown': -1.182,
 'day': 0.004,
 'duration': 0.004,
 'education=primary': -0.313,
 'education=secondary': -0.017,
 'education=tertiary': -0.029,
 'education=unknown': -0.537,
 'housing=no': -0.19,
 'housing=yes': -0.707,
 'job=admin.': -0.069,
 'job=blue-collar': -0.531,
 'job=entrepreneur': -0.57,
 'job=housemaid': 0.037,
 'job=management': 0.049,
 'job=retired': 0.522,
 'job=self-employed': 0.018,
 'job=services': -0.212,
 'job=student': 0.284,
 'job=technician': -0.341,
 'job=unemployed': -0.257,
 'job=unknown': 0.174,
 'marital=divorced': 0.054,
 'marital=married': -0.601,
 'marital=single': -0.35,
 'month=apr': 0.162,
 'month=aug': -0.714,
 'month=dec': 0.356,
 'month=feb': -0.319,
 'month=jan': -0.671,
 'month=jul': -0.796,
 'month=jun': 0.215,
 'month=mar': 0.827,
 'month=may': -0.601,
 'month=nov': -0.776,
 'month=oct': 0.931,
 'month=sep': 0.489,
 'pdays': -

In [159]:
X_train[0]

array([ 41.,  72.,   3.,   0.,   0.,   1.,   7., 764.,   0.,   0.,   1.,
         0.,   0.,   1.,   0.,   0.,   0.,   0.,   1.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   1.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   1.,   0.,   0.,   0.,  -1.,   0.,   0.,
         0.,   1.,   0.])

In [160]:
df_train

,age,job,marital,education,balance,housing,contact,day,month,duration,campaign,pdays,previous,poutcome
0,41,management,married,tertiary,72,yes,unknown,7,may,764,3,-1,0,unknown
1,38,management,married,tertiary,126,yes,unknown,21,may,164,2,-1,0,unknown
2,41,blue-collar,married,secondary,1459,no,unknown,5,jun,82,1,-1,0,unknown
3,57,services,married,secondary,162,yes,unknown,5,may,174,1,-1,0,unknown
4,30,admin.,married,secondary,124,no,telephone,16,jun,161,2,-1,0,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2707,47,admin.,single,secondary,3696,no,cellular,12,jul,250,2,181,4,success
2708,31,technician,single,secondary,1626,no,cellular,7,may,356,2,286,3,failure
2709,41,admin.,divorced,secondary,6046,yes,telephone,15,mar,300,6,182,2,success
2710,35,self-employed,married,tertiary,980,no,cellular,2,aug,428,2,336,6,failure


In [168]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


accuracy_scores = {}
accuracy_scores_diff = {}
for i in range(df_train.shape[1]):
    # Exclude the ith column by selecting all other columns

    # Convert the modified DataFrame to a list of dictionaries
    train_dict = df_train.drop(df_train.columns[i], axis=1).to_dict(orient='records')
    val_dict = df_val.drop(df_val.columns[i], axis=1).to_dict(orient='records')  # Drop the same column from validation

    # Initialize DictVectorizer
    dv = DictVectorizer(sparse=False)

    # Transform the training and validation data
    X_train = dv.fit_transform(train_dict)
    X_val = dv.transform(val_dict)

    # Print the DataFrame without the ith column
    print("Excluded column:", df_train.columns[i])
    #print("Remaining columns:", df_subset.columns.tolist())

    # Train the model
    model = LogisticRegression(max_iter=1000)  # Specify your parameters as needed
    model.fit(X_train, y_train)

    # Predict and calculate accuracy
    original_pred = model.predict(X_val)
    original_accuracy = accuracy_score(y_val, original_pred)
    accuracy_scores[f'without_{df_train.columns[i]}'] = original_accuracy
    accuracy_scores_diff[f'without_{df_train.columns[i]}'] = accuracy_model - original_accuracy
    # Output the accuracy for the current model
    #print(f"Accuracy without '{df_train.columns[i]}': {original_accuracy:.4f}")
print(accuracy_scores)
print(accuracy_scores_diff)

Excluded column: age


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Excluded column: job


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Excluded column: marital


...

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Excluded column: previous


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Excluded column: poutcome
{'without_age': 0.8849557522123894, 'without_job': 0.8915929203539823, 'without_marital': 0.8838495575221239, 'without_education': 0.8849557522123894, 'without_balance': 0.8882743362831859, 'without_housing': 0.8816371681415929, 'without_contact': 0.8805309734513275, 'without_day': 0.8827433628318584, 'without_month': 0.8871681415929203, 'without_duration': 0.8738938053097345, 'without_campaign': 0.8860619469026548, 'without_pdays': 0.8838495575221239, 'without_previous': 0.8816371681415929, 'without_poutcome': 0.8772123893805309}
{'without_age': 0.0022123893805309214, 'without_job': -0.004424778761061954, 'without_marital': 0.0033185840707964376, 'without_education': 0.0022123893805309214, 'without_balance': -0.0011061946902655162, 'without_housing': 0.00553097345132747, 'without_contact': 0.006637168141592875, 'without_day': 0.004424778761061954, 'without_month': 0.0, 'without_duration': 0.013274336283185861, 'without_campaign': 0.0011061946902655162, 'witho

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [169]:
accuracy_scores_diff

{'without_age': 0.0022123893805309214,
 'without_job': -0.004424778761061954,
 'without_marital': 0.0033185840707964376,
 'without_education': 0.0022123893805309214,
 'without_balance': -0.0011061946902655162,
 'without_housing': 0.00553097345132747,
 'without_contact': 0.006637168141592875,
 'without_day': 0.004424778761061954,
 'without_month': 0.0,
 'without_duration': 0.013274336283185861,
 'without_campaign': 0.0011061946902655162,
 'without_pdays': 0.0033185840707964376,
 'without_previous': 0.00553097345132747,
 'without_poutcome': 0.009955752212389424}

In [170]:
least_useful_feature = min(accuracy_scores_diff, key=lambda x: abs(accuracy_scores_diff[x]))
smallest_difference = accuracy_scores_diff[least_useful_feature]

print("Least useful feature based on smallest accuracy difference:", least_useful_feature)
print("Smallest difference in accuracy:", smallest_difference)

Least useful feature based on smallest accuracy difference: without_month
Smallest difference in accuracy: 0.0
